# Financial Intelligence RAG — Complete Implementation Guide
## From Jupyter Experimentation → Production Deployment on Vercel + Render

> **Your foundation:** You already built a legal RAG with BGE embeddings, BM25+vector hybrid retrieval, RRF fusion, cross-encoder reranking, semantic/hierarchical chunking, LangGraph agents, ChromaDB, and Groq. Every one of those patterns carries over. This guide maps each delta — what changes, what's new, and why.

---

## Top 3 Reference Repos (and Why)

### 1. `run-llama/sec-insights` ⭐ PRIMARY REFERENCE
**Why:** This is the only production-deployed RAG app specifically for SEC 10-K/10-Q queries, built by the LlamaIndex team. Study its document ingestion pipeline, chunking strategy, and streaming response architecture carefully. It solves the exact problem you're building — multi-document, multi-year financial queries. The production URL at secinsights.ai lets you stress-test the UX before you design your own.

**What to steal directly:**
- Document ingestion queue pattern (don't block the API while PDFs are loading)
- How they handle multi-filing queries with metadata filtering
- Their streaming response pattern for long financial answers

### 2. `siri1404/langchain-financial` ⭐ RETRIEVAL REFERENCE
**Why:** This is the most directly applicable retrieval implementation. It does exactly what your LegalRAG does but for financial docs — hybrid retriever with metadata filtering by document type (10-K vs 10-Q), fiscal year, and section (Risk Factors, MD&A). Their async batch retrieval pattern is what you need for production. Compare their retriever code against your `legal_search()` and `document_search()` functions — the diff is instructive.

**What to steal directly:**
- Metadata filtering by `doc_type`, `fiscal_year`, `section` fields on every chunk
- Async retrieval pattern (critical for Render's free tier where you want non-blocking I/O)
- Section-aware chunking for financial documents

### 3. `Abdulbasit110/nextjs-fastapi` ⭐ DEPLOYMENT REFERENCE
**Why:** Shows exactly the stack you're deploying to — Next.js on Vercel + FastAPI on Render. Study the streaming chat UI, how they handle the async server-sent events from FastAPI, and the CORS configuration between Vercel and Render. This saves you 2–3 days of debugging deployment issues.

**What to steal directly:**
- `EventSource` streaming pattern in Next.js to display token-by-token output
- FastAPI CORS middleware config for Vercel → Render cross-origin requests
- Environment variable handling across both platforms

---

## Architecture Overview

```
User types "AAPL" + "Why did operating margin compress in 2024?"
                    │
          Next.js (Vercel) — streaming UI
                    │
          FastAPI (Render) — RAG backend
           ├── SEC EDGAR API → fetch 10-K PDF
           ├── pdfplumber → table extraction
           ├── FinBERT embeddings → Qdrant Cloud
           ├── BM25 hybrid retrieval  ← your existing pattern
           ├── Cross-encoder reranking ← your existing pattern
           ├── Query decomposition (NEW)
           └── Structured answer + citations
```

The key differences from your LegalRAG:
| Component | LegalRAG (yours) | FinancialRAG (new) |
|---|---|---|
| Data source | Local PDF upload | SEC EDGAR API (auto-fetch) |
| PDF parsing | PyMuPDF (text only) | pdfplumber (text + tables) |
| Embeddings | BGE-base-en-v1.5 | ProsusAI/finbert |
| Vector store | ChromaDB (local) | Qdrant Cloud (persistent) |
| Multi-doc queries | No | Yes (multi-year filings) |
| Query handling | Rewrite + route | Decompose into sub-questions |
| Output | Text answer | Text + structured JSON table |

---

## PHASE 1: Jupyter Notebook Experimentation

### Step 1.1 — Environment Setup

```bash
# Create a fresh virtual environment
python -m venv finrag-env
source finrag-env/bin/activate  # Windows: finrag-env\Scripts\activate

pip install \
  pdfplumber \
  requests \
  qdrant-client \
  langchain \
  langchain-community \
  langchain-groq \
  langchain-huggingface \
  rank_bm25 \
  sentence-transformers \
  transformers \
  torch \
  langgraph \
  groq \
  python-dotenv \
  pandas \
  numpy \
  fastapi \
  uvicorn \
  httpx
```

Create `.env`:
```
GROQ_API_KEY=your_groq_key
QDRANT_URL=https://your-cluster.qdrant.io
QDRANT_API_KEY=your_qdrant_key
```

### Step 1.2 — SEC EDGAR Integration (New vs. LegalRAG)

This replaces your manual PDF upload. Users type a ticker; you fetch the filing automatically.

```python
import requests

def get_cik_from_ticker(ticker: str) -> str:
    """Resolve ticker symbol to SEC CIK number."""
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {"User-Agent": "YourName yourname@email.com"}  # SEC requires this
    resp = requests.get(url, headers=headers)
    data = resp.json()
    
    for entry in data.values():
        if entry["ticker"].upper() == ticker.upper():
            return str(entry["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found in SEC database")


def get_latest_10k_url(cik: str) -> str:
    """Get the URL of the most recent 10-K filing."""
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    headers = {"User-Agent": "YourName yourname@email.com"}
    resp = requests.get(url, headers=headers)
    data = resp.json()
    
    filings = data["filings"]["recent"]
    for i, form in enumerate(filings["form"]):
        if form == "10-K":
            accession = filings["accessionNumber"][i].replace("-", "")
            doc_name = filings["primaryDocument"][i]
            return f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{doc_name}"
    
    raise ValueError("No 10-K found for this company")


def download_10k(ticker: str, save_path: str = None) -> str:
    """Full pipeline: ticker → downloaded PDF path."""
    cik = get_cik_from_ticker(ticker)
    url = get_latest_10k_url(cik)
    
    headers = {"User-Agent": "YourName yourname@email.com"}
    resp = requests.get(url, headers=headers, stream=True)
    
    save_path = save_path or f"{ticker}_10K.pdf"
    with open(save_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    
    print(f"Downloaded 10-K for {ticker} → {save_path}")
    return save_path

# Test it:
# pdf_path = download_10k("AAPL")
```

**Important:** SEC rate-limits to 10 requests/second. Add `time.sleep(0.1)` between requests if fetching multiple filings.

### Step 1.3 — Table-Aware PDF Parsing (Key Upgrade from PyMuPDF)

In your LegalRAG, you used `PyMuPDFLoader`. Financial documents are full of balance sheets and income statements — pure text extraction loses the structure. Use `pdfplumber` instead.

```python
import pdfplumber
from langchain_core.documents import Document

def extract_financial_document(pdf_path: str, ticker: str, year: str) -> list[Document]:
    """
    Extracts both text and tables from a financial PDF.
    Tables are converted to structured text the LLM can reason over.
    """
    documents = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            
            # 1. Extract tables first (structured)
            tables = page.extract_tables()
            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:
                    continue
                
                # Convert table to readable text format
                # e.g., "Revenue | 2024: $391B | 2023: $383B | 2022: $394B"
                table_text = format_table_as_text(table)
                if table_text.strip():
                    documents.append(Document(
                        page_content=table_text,
                        metadata={
                            "source": pdf_path,
                            "ticker": ticker,
                            "year": year,
                            "page": page_num + 1,
                            "chunk_type": "table",
                            "table_index": table_idx,
                        }
                    ))
            
            # 2. Extract plain text (non-table regions)
            text = page.extract_text()
            if text and text.strip():
                documents.append(Document(
                    page_content=text,
                    metadata={
                        "source": pdf_path,
                        "ticker": ticker,
                        "year": year,
                        "page": page_num + 1,
                        "chunk_type": "text",
                    }
                ))
    
    print(f"Extracted {len(documents)} chunks from {ticker} {year} 10-K")
    return documents


def format_table_as_text(table: list[list]) -> str:
    """Convert a 2D table array into readable text for the LLM."""
    if not table:
        return ""
    
    rows = []
    headers = table[0]
    
    for row in table[1:]:
        if not any(cell for cell in row if cell):  # skip empty rows
            continue
        row_parts = []
        for header, cell in zip(headers, row):
            if header and cell:
                row_parts.append(f"{header}: {cell}")
        if row_parts:
            rows.append(" | ".join(row_parts))
    
    return "\n".join(rows)
```

### Step 1.4 — FinBERT Embeddings (Domain-Specific Upgrade)

Replace `BAAI/bge-base-en-v1.5` with `ProsusAI/finbert`. This is critical for financial queries — terms like "impairment", "amortization", and "liquidity risk" cluster correctly in FinBERT's embedding space in ways generic models miss.

```python
from langchain_huggingface import HuggingFaceEmbeddings
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Swap: was "BAAI/bge-base-en-v1.5", now domain-specific FinBERT
embeddings = HuggingFaceEmbeddings(
    model_name="ProsusAI/finbert",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

# Keep the same cross-encoder — bge-reranker-base still works well here
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("BAAI/bge-reranker-base")

print(f"FinBERT embeddings loaded on {device}")
```

**Note for Render deployment:** FinBERT is ~500MB. On Render's free tier, use the API-based approach instead of loading the model in-process — see Phase 3 for how to handle this.

### Step 1.5 — Qdrant Cloud (Replace ChromaDB)

Your LegalRAG wiped and rebuilt ChromaDB on every session. For financial data, you want **persistent storage** — re-indexing a 300-page 10-K takes 2–3 minutes and you shouldn't do it on every request.

First, sign up at cloud.qdrant.io (free tier: 1GB, 1 collection). Get your cluster URL and API key.

```python
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_community.vectorstores import Qdrant

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "financial_rag"
VECTOR_DIM = 768  # FinBERT output dimension

def init_qdrant_collection():
    """Create Qdrant collection if it doesn't exist."""
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    
    existing = [c.name for c in client.get_collections().collections]
    if COLLECTION_NAME not in existing:
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
        )
        print(f"Created Qdrant collection: {COLLECTION_NAME}")
    else:
        print(f"Collection {COLLECTION_NAME} already exists — reusing")
    
    return client


def build_vector_store(documents: list[Document], ticker: str, year: str):
    """Upsert documents into Qdrant with metadata filtering support."""
    client = init_qdrant_collection()
    
    vectorstore = Qdrant(
        client=client,
        collection_name=COLLECTION_NAME,
        embeddings=embeddings,
    )
    
    # Add metadata to each doc before inserting
    for doc in documents:
        doc.metadata.update({"ticker": ticker, "year": year})
    
    vectorstore.add_documents(documents)
    print(f"Upserted {len(documents)} chunks for {ticker} {year}")
    return vectorstore


def check_already_indexed(ticker: str, year: str) -> bool:
    """Check Qdrant to avoid re-indexing the same filing."""
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    
    # Scroll for any chunk matching this ticker+year
    results, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter={"must": [
            {"key": "ticker", "match": {"value": ticker}},
            {"key": "year", "match": {"value": year}},
        ]},
        limit=1,
    )
    return len(results) > 0
```

### Step 1.6 — Hierarchical Chunking with Financial Metadata

Adapt your existing `hierarchical_split()` to add financial section metadata. This allows metadata-filtered retrieval (e.g., "only search MD&A sections").

```python
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

FINANCIAL_SECTIONS = [
    "Risk Factors", "Management's Discussion", "MD&A",
    "Results of Operations", "Liquidity", "Balance Sheet",
    "Income Statement", "Cash Flow", "Notes to Financial"
]

def detect_section(text: str) -> str:
    """Detect which financial section a chunk belongs to."""
    text_lower = text.lower()
    for section in FINANCIAL_SECTIONS:
        if section.lower() in text_lower:
            return section
    return "General"

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,   # Slightly larger than LegalRAG (500) for financial context
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " "],
)

def financial_hierarchical_split(docs: list[Document]) -> tuple[list, list]:
    """
    Same parent-child pattern as your LegalRAG but with financial metadata.
    Tables are kept as atomic chunks — never split mid-table.
    """
    parent_docs, child_docs = [], []
    
    for doc in docs:
        # Tables: keep as single atomic chunks — don't split them
        if doc.metadata.get("chunk_type") == "table":
            child_docs.append(doc)  # tables go directly as child chunks
            continue
        
        # Text: same semantic → recursive pattern as your LegalRAG
        parents = semantic_chunk(doc.page_content)
        for idx, parent_text in enumerate(parents):
            section = detect_section(parent_text)
            parent_meta = {
                **doc.metadata,
                "chunk_type": "parent",
                "parent_index": idx,
                "section": section,
            }
            parent_docs.append(Document(page_content=parent_text, metadata=parent_meta))
            
            children = splitter.split_text(parent_text)
            for child_text in children:
                child_docs.append(Document(
                    page_content=child_text,
                    metadata={
                        **doc.metadata,
                        "chunk_type": "child",
                        "section": section,
                        "parent_chunk": parent_text[:200],
                    }
                ))
    
    return parent_docs, child_docs
```

### Step 1.7 — Hybrid Retrieval with Metadata Filtering

This is the same BM25 + vector + RRF pattern from your LegalRAG, but now with metadata filters to scope queries to specific tickers/years.

```python
from qdrant_client.http.models import Filter, FieldCondition, MatchValue

def financial_vector_search(query: str, ticker: str = None, year: str = None, k: int = 10):
    """Vector search with optional metadata filtering."""
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    
    # Build filter conditions
    must_conditions = []
    if ticker:
        must_conditions.append(FieldCondition(key="ticker", match=MatchValue(value=ticker)))
    if year:
        must_conditions.append(FieldCondition(key="year", match=MatchValue(value=year)))
    
    query_filter = Filter(must=must_conditions) if must_conditions else None
    
    # Embed the query
    query_vector = embeddings.embed_query(query)
    
    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        query_filter=query_filter,
        limit=k,
    )
    
    return [
        Document(
            page_content=r.payload.get("page_content", ""),
            metadata=r.payload
        )
        for r in results
    ]


# BM25 stays in-memory (same as your LegalRAG)
# Build it fresh from the chunks that match the ticker/year you're querying
def build_bm25_for_filing(ticker: str, year: str):
    """Pull all chunks for a filing from Qdrant and build BM25 index."""
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    
    results, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter={"must": [
            {"key": "ticker", "match": {"value": ticker}},
            {"key": "year", "match": {"value": year}},
        ]},
        limit=5000,  # 10-K has ~1000-3000 chunks
        with_payload=True,
    )
    
    corpus = [r.payload.get("page_content", "") for r in results]
    bm25 = BM25Okapi([c.split() for c in corpus])
    return bm25, corpus


def hybrid_financial_search(query: str, ticker: str, year: str, k: int = 10):
    """Same RRF pattern as your LegalRAG but scoped to a specific filing."""
    vector_docs = financial_vector_search(query, ticker=ticker, year=year, k=k)
    
    bm25, corpus = build_bm25_for_filing(ticker, year)
    scores = bm25.get_scores(query.split())
    idx = np.argsort(scores)[::-1][:k]
    bm25_docs = [Document(page_content=corpus[i]) for i in idx]
    
    fused = rrf_fusion(vector_docs, bm25_docs)
    return rerank(query, fused, top_k=6)  # same cross-encoder reranker as LegalRAG
```

### Step 1.8 — Query Decomposition (New vs. LegalRAG)

This is the biggest functional addition. Multi-hop financial questions ("Compare Apple and Microsoft cloud revenue 2022–2024") need to be broken into atomic sub-questions before retrieval. Your LegalRAG had `rewrite_query()` — this replaces it for complex queries.

```python
import json
from langchain_groq import ChatGroq

main_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
routing_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_tokens=50)

def is_multi_hop(question: str) -> bool:
    """Use fast LLM to classify: single-hop vs multi-hop question."""
    prompt = f"""Is this financial question multi-hop (requires comparing across years, companies, or multiple data points)?
Question: {question}
Answer only YES or NO."""
    result = routing_llm.invoke(prompt).content.strip().upper()
    return "YES" in result


def decompose_query(question: str) -> list[str]:
    """Break complex financial question into atomic sub-questions."""
    prompt = f"""You are a financial analyst. Decompose this complex question into 2-4 atomic sub-questions that can each be answered from a single SEC filing.

Complex question: {question}

Return ONLY a JSON array of sub-question strings. Example:
["What was Apple's cloud revenue in 2024?", "What was Apple's cloud revenue in 2023?", "What factors contributed to the change?"]

JSON array:"""
    
    response = main_llm.invoke(prompt).content.strip()
    
    # Parse JSON safely
    try:
        # Strip markdown fences if present
        clean = response.replace("```json", "").replace("```", "").strip()
        sub_questions = json.loads(clean)
        return sub_questions if isinstance(sub_questions, list) else [question]
    except json.JSONDecodeError:
        return [question]  # fallback: treat as single question


def extract_structured_numbers(answer: str, question: str) -> dict:
    """Extract numerical data from the answer as structured JSON."""
    prompt = f"""Extract all numerical financial data from this answer as a JSON object.
Question: {question}
Answer: {answer}

Return a JSON object with keys being metric names and values being the numbers.
If no numbers found, return {{}}.
Only return valid JSON, nothing else."""
    
    response = main_llm.invoke(prompt).content.strip()
    try:
        clean = response.replace("```json", "").replace("```", "").strip()
        return json.loads(clean)
    except:
        return {}
```

### Step 1.9 — LangGraph Agentic Pipeline

Adapt your existing `LegalState` and agent pattern. The graph is similar but the nodes are different.

```python
from typing import TypedDict, Optional, List, Any
from langgraph.graph import StateGraph, END

class FinancialState(TypedDict):
    question: str
    ticker: str
    year: str
    is_multi_hop: Optional[bool]
    sub_questions: Optional[List[str]]
    retrieved_docs: Optional[List]       # all retrieved chunks
    sub_answers: Optional[List[str]]     # answers for each sub-question
    final_answer: Optional[str]
    structured_data: Optional[dict]      # extracted numbers for UI tables
    citations: Optional[List[str]]

# --- NODES ---

def classify_node(state: FinancialState) -> dict:
    """Classify: is this multi-hop?"""
    multi_hop = is_multi_hop(state["question"])
    if multi_hop:
        sub_qs = decompose_query(state["question"])
    else:
        sub_qs = [state["question"]]
    return {"is_multi_hop": multi_hop, "sub_questions": sub_qs}


def retrieve_node(state: FinancialState) -> dict:
    """Retrieve docs for each sub-question."""
    all_docs = []
    for sub_q in state["sub_questions"]:
        docs = hybrid_financial_search(
            query=sub_q,
            ticker=state["ticker"],
            year=state["year"],
        )
        all_docs.extend(docs)
    
    # Deduplicate by content
    seen = set()
    unique_docs = []
    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)
    
    return {"retrieved_docs": unique_docs}


def answer_node(state: FinancialState) -> dict:
    """Generate the final answer with citations."""
    context = "\n\n---\n\n".join([
        f"[Source: {doc.metadata.get('ticker', '')} {doc.metadata.get('year', '')} | Page {doc.metadata.get('page', '?')} | Section: {doc.metadata.get('section', 'General')}]\n{doc.page_content}"
        for doc in state["retrieved_docs"]
    ])
    
    prompt = f"""You are a financial analyst. Answer the question using ONLY the provided context from SEC filings.
For every claim, cite the source (ticker, year, page).
If numbers are present, be precise.

Question: {state["question"]}

Context from SEC filings:
{context}

Provide a thorough answer with citations:"""
    
    answer = main_llm.invoke(prompt).content
    structured = extract_structured_numbers(answer, state["question"])
    
    citations = list(set([
        f"{doc.metadata.get('ticker')} {doc.metadata.get('year')} p.{doc.metadata.get('page', '?')}"
        for doc in state["retrieved_docs"]
    ]))
    
    return {"final_answer": answer, "structured_data": structured, "citations": citations}


# --- BUILD GRAPH ---

def build_financial_graph():
    graph = StateGraph(FinancialState)
    graph.add_node("classify", classify_node)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("answer", answer_node)
    
    graph.set_entry_point("classify")
    graph.add_edge("classify", "retrieve")
    graph.add_edge("retrieve", "answer")
    graph.add_edge("answer", END)
    
    return graph.compile()

financial_rag = build_financial_graph()


def run_financial_rag(question: str, ticker: str, year: str = "2024"):
    """Full pipeline entry point."""
    # Check if filing is already indexed; if not, download and index it
    if not check_already_indexed(ticker, year):
        print(f"Indexing {ticker} {year} 10-K...")
        pdf_path = download_10k(ticker)
        raw_docs = extract_financial_document(pdf_path, ticker, year)
        _, child_docs = financial_hierarchical_split(raw_docs)
        build_vector_store(child_docs, ticker, year)
    
    result = financial_rag.invoke({
        "question": question,
        "ticker": ticker,
        "year": year,
    })
    
    return {
        "answer": result["final_answer"],
        "structured_data": result.get("structured_data", {}),
        "citations": result.get("citations", []),
        "is_multi_hop": result.get("is_multi_hop", False),
    }

# Test in notebook:
# result = run_financial_rag("What was Apple's revenue growth in 2024?", "AAPL", "2024")
# print(result["answer"])
# print(result["structured_data"])
```

### Step 1.10 — Notebook Evaluation

Test your pipeline on 5–10 questions before moving to deployment:

```python
test_cases = [
    {"ticker": "AAPL", "year": "2024", "question": "What was Apple's total revenue in fiscal 2024?"},
    {"ticker": "AAPL", "year": "2024", "question": "What are the main risk factors Apple disclosed?"},
    {"ticker": "MSFT", "year": "2024", "question": "How did Microsoft's cloud revenue grow year over year?"},
    {"ticker": "AAPL", "year": "2024", "question": "Compare Apple's gross margin in 2024 vs 2023"},  # multi-hop
]

for case in test_cases:
    print(f"\n{'='*60}")
    print(f"Q: {case['question']}")
    result = run_financial_rag(case["question"], case["ticker"], case["year"])
    print(f"A: {result['answer'][:300]}...")
    print(f"Numbers extracted: {result['structured_data']}")
    print(f"Multi-hop: {result['is_multi_hop']}")
```

---

## PHASE 2: FastAPI Backend

### Step 2.1 — Project Structure

```
financial-rag/
├── backend/
│   ├── main.py              # FastAPI app
│   ├── rag_pipeline.py      # All your notebook code, refactored
│   ├── edgar.py             # SEC EDGAR fetching
│   ├── chunking.py          # Table extraction + hierarchical split
│   ├── retrieval.py         # Qdrant + BM25 + hybrid search
│   ├── agents.py            # LangGraph graph definition
│   ├── requirements.txt
│   └── .env
├── frontend/
│   ├── app/
│   │   ├── page.tsx         # Main chat UI
│   │   └── api/
│   │       └── chat/
│   │           └── route.ts # Next.js API route → FastAPI proxy
│   ├── package.json
│   └── .env.local
└── README.md
```

### Step 2.2 — `backend/main.py`

```python
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import asyncio
import json
import os

app = FastAPI(title="Financial RAG API")

# CORS — allow your Vercel frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",
        "https://your-app.vercel.app",  # update this after deploying frontend
        "*",  # allow all during development; tighten before production
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class QueryRequest(BaseModel):
    ticker: str
    year: str = "2024"
    question: str

class IndexRequest(BaseModel):
    ticker: str
    year: str = "2024"


@app.get("/health")
async def health():
    return {"status": "ok"}


@app.post("/index")
async def index_filing(req: IndexRequest):
    """Trigger indexing of a specific SEC filing."""
    try:
        already = check_already_indexed(req.ticker, req.year)
        if already:
            return {"status": "already_indexed", "ticker": req.ticker, "year": req.year}
        
        pdf_path = download_10k(req.ticker)
        raw_docs = extract_financial_document(pdf_path, req.ticker, req.year)
        _, child_docs = financial_hierarchical_split(raw_docs)
        build_vector_store(child_docs, req.ticker, req.year)
        
        return {"status": "indexed", "chunks": len(child_docs)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/query")
async def query(req: QueryRequest):
    """Single query → full answer (non-streaming)."""
    try:
        result = run_financial_rag(req.question, req.ticker, req.year)
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/stream")
async def stream_query(req: QueryRequest):
    """Streaming query — sends tokens as server-sent events."""
    async def generate():
        # Send a "processing" event first
        yield f"data: {json.dumps({'type': 'status', 'message': 'Searching SEC filings...'})}\n\n"
        await asyncio.sleep(0)
        
        # Run the pipeline (non-streaming internally; wrap in async)
        result = await asyncio.to_thread(
            run_financial_rag, req.question, req.ticker, req.year
        )
        
        # Stream the answer word by word for UI effect
        words = result["answer"].split()
        for i, word in enumerate(words):
            chunk = word + (" " if i < len(words) - 1 else "")
            yield f"data: {json.dumps({'type': 'token', 'content': chunk})}\n\n"
            await asyncio.sleep(0.02)
        
        # Send structured data at the end
        yield f"data: {json.dumps({'type': 'done', 'structured_data': result['structured_data'], 'citations': result['citations']})}\n\n"
    
    return StreamingResponse(generate(), media_type="text/event-stream")
```

### Step 2.3 — `backend/requirements.txt`

```
fastapi==0.111.0
uvicorn[standard]==0.30.0
pydantic==2.7.0
langchain==0.2.0
langchain-community==0.2.0
langchain-groq==0.1.0
langchain-huggingface==0.0.3
langchain-qdrant==0.1.0
langgraph==0.1.0
qdrant-client==1.9.0
sentence-transformers==3.0.0
rank_bm25==0.2.2
pdfplumber==0.11.0
requests==2.32.0
python-dotenv==1.0.0
torch==2.3.0
transformers==4.41.0
numpy==1.26.0
httpx==0.27.0
```

**Render free tier note:** Torch is heavy (~2GB). To stay within Render's 512MB RAM on the free tier, use the Groq API for embeddings via their hosted endpoint, or switch FinBERT to run as a HuggingFace Inference API call rather than loading the model locally.

Alternative lightweight embedding approach for Render:
```python
# Instead of loading FinBERT locally, call HuggingFace Inference API (free tier)
import requests

HF_API_KEY = os.getenv("HF_API_KEY")  # free at huggingface.co

def embed_via_api(texts: list[str]) -> list[list[float]]:
    url = "https://api-inference.huggingface.co/models/ProsusAI/finbert"
    headers = {"Authorization": f"Bearer {HF_API_KEY}"}
    response = requests.post(url, headers=headers, json={"inputs": texts})
    return response.json()  # returns list of embedding vectors
```

---

## PHASE 3: Render Deployment (Backend)

### Step 3.1 — Prepare for Render

1. Push your `backend/` folder to a GitHub repo (can be a subdirectory of the monorepo).

2. Create a `render.yaml` in the backend directory:
```yaml
services:
  - type: web
    name: financial-rag-backend
    env: python
    plan: free
    buildCommand: pip install -r requirements.txt
    startCommand: uvicorn main:app --host 0.0.0.0 --port $PORT
    envVars:
      - key: GROQ_API_KEY
        sync: false   # will set this in the Render dashboard
      - key: QDRANT_URL
        sync: false
      - key: QDRANT_API_KEY
        sync: false
      - key: HF_API_KEY
        sync: false
```

### Step 3.2 — Deploy on Render

1. Go to render.com → New → Web Service
2. Connect your GitHub repo
3. Set **Root Directory** to `backend/`
4. Set **Build Command:** `pip install -r requirements.txt`
5. Set **Start Command:** `uvicorn main:app --host 0.0.0.0 --port $PORT`
6. Under **Environment Variables**, add:
   - `GROQ_API_KEY`
   - `QDRANT_URL`
   - `QDRANT_API_KEY`
   - `HF_API_KEY`
7. Click **Create Web Service**

Render will give you a URL like: `https://financial-rag-backend.onrender.com`

**Free tier caveats:**
- Cold starts: the service spins down after 15 minutes of inactivity. First request after sleep takes ~30 seconds. Add a loading spinner in your frontend.
- 512MB RAM: if you load FinBERT locally, it'll OOM. Use the HuggingFace Inference API approach above.
- 0.1 CPU: indexing a 300-page 10-K will take 5–8 minutes. Design your UX around async indexing — trigger indexing in the background and poll for status.

### Step 3.3 — Health Check

After deploying, test:
```bash
curl https://financial-rag-backend.onrender.com/health
# Expected: {"status": "ok"}

curl -X POST https://financial-rag-backend.onrender.com/query \
  -H "Content-Type: application/json" \
  -d '{"ticker": "AAPL", "year": "2024", "question": "What was Apple revenue?"}'
```

---

## PHASE 4: Next.js Frontend on Vercel

### Step 4.1 — Setup Next.js

```bash
cd frontend/
npx create-next-app@latest . --typescript --tailwind --app
npm install eventsource-parser  # for SSE parsing
```

### Step 4.2 — `frontend/app/page.tsx` (Main Chat UI)

```tsx
"use client";

import { useState, useRef, useEffect } from "react";

const BACKEND_URL = process.env.NEXT_PUBLIC_BACKEND_URL || "http://localhost:8000";

interface Message {
  role: "user" | "assistant";
  content: string;
  structuredData?: Record<string, any>;
  citations?: string[];
}

export default function Home() {
  const [ticker, setTicker] = useState("AAPL");
  const [year, setYear] = useState("2024");
  const [question, setQuestion] = useState("");
  const [messages, setMessages] = useState<Message[]>([]);
  const [loading, setLoading] = useState(false);
  const [indexing, setIndexing] = useState(false);
  const [statusMsg, setStatusMsg] = useState("");
  const answerRef = useRef("");

  const indexFiling = async () => {
    setIndexing(true);
    setStatusMsg(`Fetching and indexing ${ticker} ${year} 10-K from SEC EDGAR...`);
    try {
      const res = await fetch(`${BACKEND_URL}/index`, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ ticker, year }),
      });
      const data = await res.json();
      setStatusMsg(data.status === "already_indexed" 
        ? `${ticker} ${year} already indexed ✓` 
        : `Indexed ${data.chunks} chunks for ${ticker} ${year} ✓`
      );
    } catch (err) {
      setStatusMsg("Indexing failed — check backend");
    }
    setIndexing(false);
  };

  const sendMessage = async () => {
    if (!question.trim()) return;
    
    const userMsg: Message = { role: "user", content: question };
    setMessages(prev => [...prev, userMsg]);
    setQuestion("");
    setLoading(true);
    answerRef.current = "";
    
    // Add placeholder assistant message
    setMessages(prev => [...prev, { role: "assistant", content: "" }]);
    
    try {
      const res = await fetch(`${BACKEND_URL}/stream`, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ ticker, year, question: userMsg.content }),
      });
      
      const reader = res.body?.getReader();
      const decoder = new TextDecoder();
      
      while (true) {
        const { done, value } = await reader!.read();
        if (done) break;
        
        const text = decoder.decode(value);
        const lines = text.split("\n");
        
        for (const line of lines) {
          if (!line.startsWith("data: ")) continue;
          try {
            const event = JSON.parse(line.slice(6));
            
            if (event.type === "status") {
              setStatusMsg(event.message);
            } else if (event.type === "token") {
              answerRef.current += event.content;
              setMessages(prev => {
                const updated = [...prev];
                updated[updated.length - 1] = {
                  role: "assistant",
                  content: answerRef.current,
                };
                return updated;
              });
            } else if (event.type === "done") {
              setMessages(prev => {
                const updated = [...prev];
                updated[updated.length - 1] = {
                  ...updated[updated.length - 1],
                  structuredData: event.structured_data,
                  citations: event.citations,
                };
                return updated;
              });
              setStatusMsg("");
            }
          } catch {}
        }
      }
    } catch (err) {
      setStatusMsg("Request failed");
    }
    
    setLoading(false);
  };

  return (
    <main className="min-h-screen bg-gray-950 text-white flex flex-col">
      {/* Header */}
      <div className="border-b border-gray-800 p-4 flex items-center gap-4">
        <h1 className="text-xl font-bold text-blue-400">Financial RAG</h1>
        <input
          value={ticker}
          onChange={e => setTicker(e.target.value.toUpperCase())}
          placeholder="Ticker (e.g. AAPL)"
          className="bg-gray-800 border border-gray-700 rounded px-3 py-1 text-sm w-28"
        />
        <select
          value={year}
          onChange={e => setYear(e.target.value)}
          className="bg-gray-800 border border-gray-700 rounded px-3 py-1 text-sm"
        >
          {["2024", "2023", "2022", "2021"].map(y => (
            <option key={y}>{y}</option>
          ))}
        </select>
        <button
          onClick={indexFiling}
          disabled={indexing}
          className="bg-blue-600 hover:bg-blue-700 disabled:opacity-50 px-3 py-1 rounded text-sm"
        >
          {indexing ? "Indexing..." : "Load Filing"}
        </button>
        {statusMsg && <span className="text-gray-400 text-sm">{statusMsg}</span>}
      </div>

      {/* Chat area */}
      <div className="flex-1 overflow-y-auto p-4 space-y-4 max-w-3xl mx-auto w-full">
        {messages.map((msg, i) => (
          <div key={i} className={`flex ${msg.role === "user" ? "justify-end" : "justify-start"}`}>
            <div className={`max-w-[80%] rounded-lg p-3 ${
              msg.role === "user" ? "bg-blue-700" : "bg-gray-800"
            }`}>
              <p className="whitespace-pre-wrap text-sm">{msg.content}</p>
              
              {/* Structured data table */}
              {msg.structuredData && Object.keys(msg.structuredData).length > 0 && (
                <div className="mt-3 border border-gray-600 rounded overflow-hidden">
                  <table className="w-full text-xs">
                    <thead className="bg-gray-700">
                      <tr>
                        <th className="text-left p-2">Metric</th>
                        <th className="text-right p-2">Value</th>
                      </tr>
                    </thead>
                    <tbody>
                      {Object.entries(msg.structuredData).map(([k, v]) => (
                        <tr key={k} className="border-t border-gray-700">
                          <td className="p-2 text-gray-300">{k}</td>
                          <td className="p-2 text-right font-mono text-green-400">{String(v)}</td>
                        </tr>
                      ))}
                    </tbody>
                  </table>
                </div>
              )}
              
              {/* Citations */}
              {msg.citations && msg.citations.length > 0 && (
                <div className="mt-2 text-xs text-gray-400">
                  Sources: {msg.citations.join(" · ")}
                </div>
              )}
            </div>
          </div>
        ))}
      </div>

      {/* Input area */}
      <div className="border-t border-gray-800 p-4 max-w-3xl mx-auto w-full">
        <div className="flex gap-2">
          <input
            value={question}
            onChange={e => setQuestion(e.target.value)}
            onKeyDown={e => e.key === "Enter" && !e.shiftKey && sendMessage()}
            placeholder={`Ask about ${ticker} ${year} financials...`}
            className="flex-1 bg-gray-800 border border-gray-700 rounded px-3 py-2 text-sm"
            disabled={loading}
          />
          <button
            onClick={sendMessage}
            disabled={loading || !question.trim()}
            className="bg-blue-600 hover:bg-blue-700 disabled:opacity-50 px-4 py-2 rounded text-sm"
          >
            {loading ? "..." : "Ask"}
          </button>
        </div>
      </div>
    </main>
  );
}
```

### Step 4.3 — `frontend/.env.local`

```
NEXT_PUBLIC_BACKEND_URL=https://financial-rag-backend.onrender.com
```

### Step 4.4 — Deploy to Vercel

```bash
# Install Vercel CLI
npm install -g vercel

cd frontend/
vercel

# Follow prompts:
# - Link to existing project? No
# - Project name: financial-rag
# - Framework: Next.js (auto-detected)
```

After first deploy, set environment variable in Vercel dashboard:
- `NEXT_PUBLIC_BACKEND_URL` = your Render backend URL

Then redeploy:
```bash
vercel --prod
```

---

## PHASE 5: Free Platform Configuration Summary

| Service | What it hosts | Free tier limits | What to watch |
|---|---|---|---|
| **Vercel** | Next.js frontend | 100GB bandwidth/mo, unlimited deploys | None for this project |
| **Render** | FastAPI backend | 750 hrs/mo, 512MB RAM, spins down | Cold starts, RAM for models |
| **Qdrant Cloud** | Vector store | 1GB storage, 1 collection | ~5,000–10,000 chunks max |
| **Groq** | LLM (Llama 3) | 30 req/min, 14,400 req/day | Rate limit on bulk evals |
| **HuggingFace** | FinBERT embeddings | Inference API free tier | Cold starts on HF side |

**Staying within free limits:**
- Keep only 2–3 tickers indexed simultaneously in Qdrant
- Cache frequently-asked questions in memory (simple dict in FastAPI)
- Add a loading state for Render cold starts (frontend shows "Waking up backend...")

---

## PHASE 6: What to Tell Recruiters

> "Generic RAG fails on SEC filings because financial questions are inherently multi-hop — comparing operating margins across three annual reports while reasoning over tables isn't retrieval, it's multi-document analysis. I implemented query decomposition to break complex questions into atomic sub-questions, domain-specific FinBERT embeddings instead of generic sentence-transformers so financial terms like 'impairment' and 'goodwill write-down' cluster correctly in embedding space, and pdfplumber for table-aware chunking so balance sheets are preserved as structured data rather than garbled text. The SEC EDGAR API integration means users type a ticker symbol and the system automatically fetches, parses, and indexes the filing — no manual PDF uploads. Built on FastAPI + Qdrant Cloud + Next.js, deployed across Render and Vercel."

---

## Key Differences from Your LegalRAG — Quick Reference

| LegalRAG (your notebook) | FinancialRAG (this guide) | Why it changed |
|---|---|---|
| `PyMuPDFLoader` | `pdfplumber` | Tables in financial docs need structured extraction |
| `BAAI/bge-base-en-v1.5` | `ProsusAI/finbert` | Domain-specific embeddings for financial language |
| ChromaDB (local, wiped each run) | Qdrant Cloud (persistent) | Don't re-index 300-page PDFs on every request |
| Manual PDF upload | SEC EDGAR API auto-fetch | What makes it feel like a real product |
| `rewrite_query()` | `decompose_query()` | Multi-hop questions need sub-questions, not just rewrites |
| Single-document | Multi-document + metadata filters | Cross-year comparison queries |
| Text answer only | Text + structured JSON table | UI can render comparison tables automatically |
| Chroma global state | Qdrant with ticker/year metadata | Scope retrieval to the right filing |

Everything else — BM25, RRF, cross-encoder reranking, LangGraph graph structure, Groq/Llama, semantic chunking — carries over directly from your existing notebook.
